# LAB | First match ≠ right match
## Relevance Scoring & Rerankers for Trustworthy AI + EU AI Act

**Learning goal:** build one retrieval path over two sources and **prove** every answer is grounded in evidence you can inspect.

**Your sources**
- `podcast.txt`: a podcast transcript about the EU *Ethics Guidelines for Trustworthy AI* (2019)
- `eu_ai_act.pdf`: Regulation (EU) 2024/1689, the AI Act (144 pages)

**What you submit:** a `lab_proof.md` containing your code (or a link to this notebook), **one query**, the **retrieved evidence**, the **final answer**, and **one failure or limitation**. There's a template at the end of this notebook.

---

### How this notebook works

| Label | Meaning |
|---|---|
| **PROVIDED** | Code is given. Run it and read it. |
| **YOUR TURN** | Replace the `# TODO` lines with your own code. |
| **OPTIONAL (Advanced)** | Do this if you have time; it's not required to pass. |

Each step ends with a **Done when** check. Don't move on until it passes.

### Prompt discipline
You may use AI to **review, debug or critique** your work, but not to write your final answer or your `lab_proof.md`. A good review prompt:

> *Role: Retrieval reviewer. Task: Review my plan for grounding and verification. Inputs: my sources, query, retrieval path, and evidence. Constraints: Do not produce the final answer. Output: three grounding risks, two trace checks, one simplification.*

---
## Step 0: Environment [PROVIDED]

1. Create a `.env` file next to this notebook containing `OPENAI_API_KEY=...` and `PINECONE_API_KEY=...`
2. Create a folder called `data/` next to this notebook and put `podcast.txt` and `eu_ai_act.pdf` inside it.
3. Run the two cells below.

**Warning:** Never paste API keys into a notebook cell, and never commit your `.env` file.

In [6]:
%pip install -U -q python-dotenv pypdf langchain-core langchain-community langchain-text-splitters langchain-openai pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    if not os.getenv(key):
        raise ValueError(f"{key} not found. Add it to your .env file.")
print("API keys loaded")

API keys loaded


---
## Step 1: Setup & Data Preparation [PROVIDED, except 1d]

**Goal:** turn two very different sources into chunks that each carry **metadata**, meaning labels that say *where the chunk came from*.

**Why metadata matters:** when the system answers *"Fines are up to €35M"*, a reviewer must be able to trace that sentence back to *"EU AI Act, page 115"*. Without metadata you can't trace an answer, and you can't filter results either (Step 5).

Parts 1a–1c are done for you: run each cell and **read the output**. Part 1d (embeddings) is yours to write.

### 1a. Load both sources

In [8]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader

DATA_DIR = Path("data")
PODCAST_PATH = DATA_DIR / "podcast.txt"
PDF_PATH = DATA_DIR / "eu_ai_act.pdf"

for path in [PODCAST_PATH, PDF_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the lab files in a 'data' folder next to this notebook.")

podcast_docs = TextLoader(str(PODCAST_PATH), encoding="utf-8").load()  # 1 Document (whole transcript)
pdf_docs = PyPDFLoader(str(PDF_PATH)).load()                           # 1 Document per PDF page

print(f"Podcast: {len(podcast_docs)} document, {len(podcast_docs[0].page_content):,} characters")
print(f"EU AI Act: {len(pdf_docs)} pages")
print("\nRaw metadata from the PDF loader:", pdf_docs[0].metadata)

/var/folders/z7/nj2t4t2x5677_xvdkhy6hrdr0000gn/T/ipykernel_48308/320955083.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


Podcast: 1 document, 16,414 characters
EU AI Act: 144 pages

Raw metadata from the PDF loader: {'producer': 'PDFlib+PDI 9.0.7p3 (C++/Win64)', 'creator': 'Servigistics Arbortext Publishing Engine', 'creationdate': '2024-07-11T14:47:17+02:00', 'author': 'Publications Office of the European Union L-2985 Luxembourg LUXEMBOURG', 'keywords': 'ISSN 1977-0677', 'moddate': '2024-07-11T15:55:28+02:00', 'subject': 'I Legislative acts', 'title': 'Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)Text with EEA relevance.', 'source': 'data/eu_ai_act.pdf', 'total_pages': 144, 'page': 0, 'page_label': '1'}


### 1b. Chunk with metadata

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

def chunk_with_metadata(docs, source_type, source_name):
    """Split documents into chunks and give every chunk a clean, consistent metadata schema."""
    chunks = splitter.split_documents(docs)
    for i, chunk in enumerate(chunks):
        metadata = {
            "source_type": source_type,        # "podcast" or "regulation" -> used for filtering in Step 5
            "source_name": source_name,        # human-readable name -> used for citations
            "chunk_id": f"{source_type}-{i}",  # unique id -> lets a reviewer trace an exact chunk
        }
        # PDF pages are 0-indexed in the loader; store the page number a human would see
        if "page" in chunk.metadata:
            metadata["page"] = int(chunk.metadata["page"]) + 1
        chunk.metadata = metadata
    return chunks

podcast_chunks = chunk_with_metadata(podcast_docs, "podcast", "Trustworthy AI podcast")
act_chunks = chunk_with_metadata(pdf_docs, "regulation", "EU AI Act (Reg. 2024/1689)")
all_chunks = podcast_chunks + act_chunks

print(f"Podcast chunks:   {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(act_chunks)}")
print(f"Total chunks:     {len(all_chunks)}")

Podcast chunks:   20
EU AI Act chunks: 764
Total chunks:     784


### 1c. Inspect your chunks (always look at your data!)

In [10]:
for chunk in [podcast_chunks[0], act_chunks[150]]:
    print(chunk.metadata)
    print(chunk.page_content[:300], "...")
    print("-" * 80)

{'source_type': 'podcast', 'source_name': 'Trustworthy AI podcast', 'chunk_id': 'podcast-0'}
So imagine for a second you're driving across a, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No, you just, you trust it. You just drive, you trust the bridge, you trust the ...
--------------------------------------------------------------------------------
{'source_type': 'regulation', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'chunk_id': 'regulation-150', 'page': 21}
indicators relating to AI.
OJ L, 12.7.2024 EN
ELI: http://data.europa.eu/eli/reg/2024/1689/oj 21/144
(35) Directive 2014/31/EU of the European Parliament and of the Council of 26 February 2014 on the harmonisation of the laws of the 
Member States relating to the making available on the market of no ...
--------------------------------------------------------------------------------


### 1d. Embeddings model [YOUR TURN]

Set up the embedding model yourself and test it on **one** chunk.

**What to do**
1. Import `OpenAIEmbeddings` from `langchain_openai`.
2. Create a variable called `embeddings` using the model `text-embedding-3-small`. Step 2 uses this exact variable name.
3. Embed the text of `all_chunks[0]` with `embeddings.embed_query(...)` and print the length of the vector.

**Hint:** see **Section 2 of `rag_intro.ipynb`**.
**Note:** you don't embed **all** chunks here. That happens in Step 2, when you upload them to Pinecone, so you only pay for them once.

In [11]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

test_embedding = embeddings.embed_query(all_chunks[0].page_content)
print(f"Test embedding vector length: {len(test_embedding)}")

Test embedding vector length: 1536


### Stop and think (write your answers in `lab_proof.md` later)

1. Our metadata has `source_type`, `source_name`, `chunk_id` and `page`. **What's missing?** Think about a user asking *"What does Article 5 say?"* or *"Only show me recitals."* Could you filter on that today?
2. The podcast has no pages. How would a reviewer find where a podcast chunk came from?

You'll come back to this in **Step 5**.

**Done when:** you can see chunk counts for both sources, every chunk you print shows a `metadata` dict, and your test embedding has length **1536**.

---
## Step 2: Embeddings + Baseline Retrieval [YOUR TURN]

**Goal:** build a plain vector search. This is your **baseline**: every later step must beat it, or it isn't worth the extra cost.

**What to do**
1. Create (or connect to) a Pinecone index. Use dimension `1536` and metric `cosine`.
2. Upload `all_chunks` with `PineconeVectorStore.from_documents(...)`. This is the step that embeds every chunk.
3. Write `baseline_search(query, k)`. It should return `(doc, score)` pairs using `similarity_search_with_score`.
4. Run it on the **fixed test queries** below and save the results.

**Hint:** Stuck? Look at **Section 3–4 of `rag_intro.ipynb`** (yesterday's notebook).
**Warning:** Only run the upload cell **once**. Running it again uploads duplicate chunks.

In [12]:
# PROVIDED: fixed test queries. Use the SAME queries in every step so you can compare fairly.
TEST_QUERIES = [
    "What AI practices are prohibited under the EU AI Act?",
    "What are the seven requirements for trustworthy AI?",
    "What is the difference between human-in-the-loop and human-on-the-loop?",
    "What fines apply for using a prohibited AI practice?",
    "Is social scoring of citizens allowed?",
]

# PROVIDED: helper to print results in the same format in every step
def show_results(query, results, label="Results"):
    """results: list of (Document, score) pairs"""
    print(f"\n=== {label} | Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        m = doc.metadata
        where = f"p.{m['page']}" if "page" in m else m.get("chunk_id")
        print(f"#{rank}  score={score:.3f}  [{m.get('source_type')} | {where}]  {doc.page_content[:120]!r}")

In [14]:
# YOUR TURN: create / connect to the Pinecone index
import os
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "lab-rerankers"   # use lowercase letters and dashes only

# Create the index only if it does not already exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# Upload the chunks and create the vector store
vectorstore = PineconeVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    index_name=index_name
)

In [16]:
# YOUR TURN: baseline retrieval

def baseline_search(query, k=5):
    return vectorstore.similarity_search_with_score(query, k=k)


baseline_results = {}

for q in TEST_QUERIES:
    baseline_results[q] = baseline_search(q)
    show_results(q, baseline_results[q], label="Baseline results")


=== Baseline results | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#3  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#4  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#5  score=0.682  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'

=== Baseline results | Query: What are the seven requirements for trustworthy AI?
#1  score=0

**Done when:** each test query prints 5 results with a score, a source type and a page or chunk id.

**Note it down:** for which query does the **top result** look wrong or only half-relevant? That's your *first match ≠ right match* case.

---
## Step 3: LLM Relevance Scoring [OPTIONAL (Advanced)]

**Goal:** have an LLM judge how relevant each retrieved chunk is, then re-order the results.

**What to do**
1. Retrieve a wider set of candidates (e.g. `k=10`) with `baseline_search`.
2. For each chunk, ask `gpt-4o-mini` to score its relevance to the query from 0 to 1.
3. Combine the two scores, e.g. `combined = 0.3 * similarity + 0.7 * llm_score`, and sort by `combined`.
4. Compare the new top 3 with the baseline top 3.

**Hint:** Reuse `rerank_documents()` from **Section 3 of `rag_pipeline.ipynb`**. It already parses the score and prints it.
**Warning:** This makes 1 LLM call **per chunk, per query**. Count the calls you make.

In [18]:
# Compare baseline retrieval with LLM-reranked retrieval

reranked_results = {}

for q in TEST_QUERIES:
    reranked_results[q] = llm_rerank(
        q,
        k_initial=10,
        k_final=5
    )

    print(f"\n{'=' * 80}")
    print(f"QUERY: {q}")
    print(f"{'=' * 80}")

    print("\nBASELINE RESULTS:")
    show_results(q, baseline_results[q], label="Baseline")

    print("\nLLM-RERANKED RESULTS:")
    show_results(q, reranked_results[q], label="LLM-reranked")


def document_id(doc):
    return (
        doc.metadata.get("source"),
        doc.metadata.get("page"),
        doc.metadata.get("chunk_id"),
        doc.page_content[:100]
    )


for q in TEST_QUERIES:
    baseline_order = [
        document_id(doc)
        for doc, score in baseline_results[q]
    ]

    reranked_order = [
        document_id(doc)
        for doc, score in reranked_results[q]
    ]

    if baseline_order != reranked_order:
        print(f"\nOrder changed for query:\n{q}")
        print("\nBaseline order:")
        for i, item in enumerate(baseline_order, start=1):
            print(f"{i}. {item}")

        print("\nReranked order:")
        for i, item in enumerate(reranked_order, start=1):
            print(f"{i}. {item}")

        print(
            "\nJudgment: inspect the two result lists above and decide whether "
            "the reranked order is better or worse based on relevance."
        )
        break
else:
    print("No order changed for any query.")


QUERY: What AI practices are prohibited under the EU AI Act?

BASELINE RESULTS:

=== Baseline | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#3  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#4  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#5  score=0.682  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'

LLM-RERANKED RESULTS

**Done when:** you can show at least one query where the order changed, and say whether it got better or worse.

---
## Step 4: Dedicated Reranker (Cross-Encoder or Cohere) [OPTIONAL (Advanced)]

**Goal:** use a model built specifically for reranking. It's usually faster and cheaper than an LLM, and often better.

**Option A: local cross-encoder (no API key needed)**
```
%pip install -q sentence-transformers
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = reranker.predict([(query, doc.page_content) for doc in docs])
```

**Option B: Cohere Rerank API** (needs `COHERE_API_KEY` in `.env`; see the Cohere Rerank docs)

**What to do:** rerank the same 10 baseline candidates, keep the top 5, and compare with Step 3.

In [20]:
%pip install -q sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [21]:
# STEP 4: Dedicated Cross-Encoder Reranker

from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def crossencoder_rerank(query, k_initial=10, k_final=5):
    # Retrieve initial candidates
    initial_results = baseline_search(query, k=k_initial)

    docs = [doc for doc, similarity_score in initial_results]

    # Score query-document pairs
    pairs = [
        (query, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    # Combine documents and cross-encoder scores
    reranked_results = list(zip(docs, scores))

    # Highest score first
    reranked_results.sort(
        key=lambda item: item[1],
        reverse=True
    )

    return reranked_results[:k_final]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5887.65it/s]


In [22]:
# Compare baseline vs cross-encoder reranked results

crossencoder_results = {}

for q in TEST_QUERIES[:2]:
    crossencoder_results[q] = crossencoder_rerank(
        q,
        k_initial=10,
        k_final=3
    )

    baseline_top3 = baseline_results[q][:3]

    print(f"\n{'=' * 80}")
    print(f"QUERY: {q}")
    print(f"{'=' * 80}")

    print("\nBASELINE TOP 3:")
    show_results(q, baseline_top3, label="Baseline")

    print("\nCROSS-ENCODER RERANKED TOP 3:")
    show_results(
        q,
        crossencoder_results[q],
        label="Cross-encoder reranked"
    )


QUERY: What AI practices are prohibited under the EU AI Act?

BASELINE TOP 3:

=== Baseline | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#3  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'

CROSS-ENCODER RERANKED TOP 3:

=== Cross-encoder reranked | Query: What AI practices are prohibited under the EU AI Act?
#1  score=6.939  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#2  score=6.939  [regulation | p.12.0]  '(45) Practices that are

**Done when:** you have a side-by-side of baseline vs reranked top 3 for at least two queries.

---
## Step 5: Metadata Filtering [YOUR TURN]

**Goal:** restrict search to the right part of the knowledge base *before* ranking.

**What to do**
1. Filter by source using Pinecone's filter syntax:
   `vectorstore.similarity_search_with_score(query, k=5, filter={"source_type": {"$eq": "regulation"}})`
2. Run one query twice: once with **only the regulation**, once with **only the podcast**. How do the answers differ?
3. **The real lesson:** try to filter on something your metadata *doesn't* have yet, such as `section` (`"recital"` / `"article"` / `"annex"`) or `article` (`"Article 5"`). You can't.
   → Go back to `chunk_with_metadata` in Step 1, add the field, **delete the index and re-upload** (`pc.delete_index(index_name)`), then try again.

**Hint:** Hints for the EU AI Act: in this PDF the recitals run up to about page 44, the Articles start at *"CHAPTER I"*, and the Annexes start at *"ANNEX I"*. A regex like `r"^Article\s+(\d+)"` on each line can detect article headings.
**Warning:** Pinecone metadata values can't be `None`. Leave the key out instead.

In [23]:
# YOUR TURN: filter by source_type

query = TEST_QUERIES[0]

regulation_results = vectorstore.similarity_search_with_score(
    query,
    k=5,
    filter={"source_type": {"$eq": "regulation"}}
)

podcast_results = vectorstore.similarity_search_with_score(
    query,
    k=5,
    filter={"source_type": {"$eq": "podcast"}}
)

show_results(query, regulation_results, label="EU AI Act only")
show_results(query, podcast_results, label="Podcast only")


=== EU AI Act only | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#3  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#4  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#5  score=0.682  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'

=== Podcast only | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.519

In [24]:
# Recreate chunks with section metadata

import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)


def chunk_with_metadata(docs, source_type, source_name):
    chunks = splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        metadata = {
            "source_type": source_type,
            "source_name": source_name,
            "chunk_id": f"{source_type}-{i}",
        }

        if "page" in chunk.metadata:
            metadata["page"] = int(chunk.metadata["page"]) + 1

        if source_type == "regulation":
            text = chunk.page_content.upper()

            if "ANNEX I" in text or "ANNEX II" in text:
                metadata["section"] = "annex"
            elif re.search(r"\bARTICLE\s+\d+\b", text):
                metadata["section"] = "article"
            else:
                metadata["section"] = "recital"

        chunk.metadata = metadata

    return chunks


podcast_chunks = chunk_with_metadata(
    podcast_docs,
    "podcast",
    "Trustworthy AI podcast"
)

act_chunks = chunk_with_metadata(
    pdf_docs,
    "regulation",
    "EU AI Act (Reg. 2024/1689)"
)

all_chunks = podcast_chunks + act_chunks

print(f"Podcast chunks:   {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(act_chunks)}")
print(f"Total chunks:     {len(all_chunks)}")

Podcast chunks:   20
EU AI Act chunks: 764
Total chunks:     784


In [25]:
# Rebuild the index so it contains the new metadata

pc.delete_index(index_name)

pc.create_index(
    name=index_name,
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

vectorstore = PineconeVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    index_name=index_name
)

print("Index rebuilt successfully.")

Index rebuilt successfully.


In [26]:
# YOUR TURN: filter by the new metadata field

query = "What AI practices are prohibited under the EU AI Act?"

article_results = vectorstore.similarity_search_with_score(
    query,
    k=5,
    filter={
        "$and": [
            {"source_type": {"$eq": "regulation"}},
            {"section": {"$eq": "article"}}
        ]
    }
)

show_results(
    query,
    article_results,
    label="EU AI Act articles only"
)


=== EU AI Act articles only | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.673  [regulation | p.45.0]  '(b) prohibitions of certain AI practices;\n(c) specific requirements for high-risk AI systems and obligations for operato'
#3  score=0.636  [regulation | p.46.0]  '6. This Regulation does not apply to AI systems or AI models, including their output, specifically developed and put int'
#4  score=0.623  [regulation | p.121.0]  '(EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) \n2020/1828 (Artificia'
#5  score=0.619  [regulation | p.113.0]  'Article 18 of Regulation (EU) 2019/1020 shall apply mutatis mutandis to the providers of the general-purpose AI model, \n'


**Done when:** you can run a query that only returns, e.g., *Article* chunks from the EU AI Act, and you can explain why you had to rebuild the index to do it.

---
## Step 6: Complete RAG Pipeline with Reranking [YOUR TURN]

**Goal:** plug your best retrieval method into a full RAG chain that **cites its sources**.

**What to do**
1. Build an LCEL chain: `retrieve → format context → prompt → LLM → answer` (see **Section 5 of `rag_intro.ipynb`**).
2. In `format_context`, put each chunk's source and page **inside** the context, e.g. `[EU AI Act, p.51]`, and tell the LLM to cite them.
3. Make it easy to switch retrieval methods: baseline vs. reranked should be a **1–2 line change**.
4. Make the model say *"I don't know based on the sources"* when the context doesn't contain the answer.

In [28]:
# STEP 6: Complete RAG Pipeline with Reranking

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


def format_context(results):
    formatted_chunks = []

    for i, (doc, score) in enumerate(results, start=1):
        metadata = doc.metadata
        source = metadata.get("source_name", metadata.get("source_type", "Unknown source"))

        if "page" in metadata:
            location = f"page {metadata['page']}"
        else:
            location = f"chunk {metadata.get('chunk_id', 'unknown')}"

        formatted_chunks.append(
            f"[{source}, {location}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


rag_prompt = ChatPromptTemplate.from_template(
    """
You are a question-answering assistant.

Answer the question using ONLY the information in the provided context.
Do not use outside knowledge.
Cite the source after each important claim using this format:
[source name, page number]
or:
[source name, chunk_id]

If the context does not contain enough information to answer the question, say:
"I don't know based on the sources."

Context:
{context}

Question:
{question}

Answer:
"""
)


def answer(query, retrieve_fn=baseline_search):
    # Retrieve documents
    results = retrieve_fn(query)

    # Format retrieved documents with citations
    context = format_context(results)

    # Build the RAG chain
    rag_chain = rag_prompt | llm | StrOutputParser()

    # Generate the answer
    response = rag_chain.invoke({
        "context": context,
        "question": query
    })

    return response, results

In [29]:
query = TEST_QUERIES[0]

response, results = answer(query)

print(response)
print("\nRetrieved evidence:")
show_results(query, results)

The following AI practices are prohibited under the EU AI Act:

1. The placing on the market, putting into service, or use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective or effect of materially distorting the behavior of a person or a group of persons by appreciably impairing their ability to make an informed decision [EU AI Act (Reg. 2024/1689), page 51.0].

2. Unfair commercial practices leading to economic or financial harms to consumers are prohibited under all circumstances, irrespective of whether they are implemented through AI systems or otherwise [EU AI Act (Reg. 2024/1689), page 9.0].

Retrieved evidence:

=== Results | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.684  [regulatio

In [ ]:
# YOUR TURN
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def format_context(results):
    # TODO: join chunks into one string, each prefixed with its source + page/chunk_id
    pass

rag_prompt = ChatPromptTemplate.from_template(
    # TODO: write a prompt that answers ONLY from context, cites [source, page], and admits when it doesn't know
    ""
)

def answer(query, retrieve_fn=baseline_search):   # <- swap retrieve_fn to switch methods
    # TODO: retrieve -> format -> prompt | llm | StrOutputParser() -> return answer AND the results used
    pass

In [30]:
# Step 6 verification: answer with citations and exact supporting chunks

query = "What AI practices are prohibited under the EU AI Act?"

response, used_results = answer(
    query,
    retrieve_fn=crossencoder_rerank
)

print("=" * 80)
print("QUESTION")
print(query)

print("\n" + "=" * 80)
print("ANSWER WITH CITATIONS")
print(response)

print("\n" + "=" * 80)
print("EXACT RETRIEVED CHUNKS BEHIND THE CLAIMS")

for rank, (doc, score) in enumerate(used_results, start=1):
    metadata = doc.metadata

    source = metadata.get("source_name", "Unknown source")
    location = (
        f"page {metadata['page']}"
        if "page" in metadata
        else f"chunk {metadata.get('chunk_id', 'unknown')}"
    )

    print(f"\nChunk {rank}: [{source}, {location}]")
    print(f"Score: {score:.3f}")
    print(doc.page_content)
    print("-" * 80)

QUESTION
What AI practices are prohibited under the EU AI Act?

ANSWER WITH CITATIONS
The following AI practices are prohibited under the EU AI Act:

1. The placing on the market, putting into service, or use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective or effect of materially distorting the behavior of a person or a group of persons by appreciably impairing their ability to make an informed decision [EU AI Act (Reg. 2024/1689), page 51.0].

2. AI-enabled manipulative techniques that can persuade persons to engage in unwanted behaviors or deceive them by nudging them into decisions in a way that subverts and impairs their autonomy, decision-making, and free choices, particularly when significant harms are likely to occur, impacting physical, psychological health, or financial interests [EU AI Act (Reg. 2024/1689), page 9.0].

3. Practices that contradict Union values of respec

**Done when:** one query gives an answer with citations, and you can point to the exact chunk behind each claim.

---
## Step 7: Evaluate: With vs. Without Reranking [YOUR TURN]

**Goal:** decide, with evidence, whether reranking was worth it.

**What to do:** for each test query, read the answers and the retrieved chunks yourself, then fill in the table. You are the judge: no automatic metric.

| Query | Method | Top-1 chunk relevant? (Y/N) | Answer correct? (Y/Partial/N) | Grounded in cited chunk? (Y/N) | Notes |
|---|---|---|---|---|---|
| Q1 | Baseline | | | | |
| Q1 | Reranked | | | | |
| Q2 | Baseline | | | | |
| Q2 | Reranked | | | | |
| … | | | | | |

**Then answer:**
- Did reranking change the answer, or only the order?
- What did it cost (extra calls, seconds per query)? Would a client pay for that?

In [31]:
# STEP 7: Compare baseline vs reranked answers

import time

for q in TEST_QUERIES:
    print("\n" + "=" * 100)
    print(f"QUERY: {q}")
    print("=" * 100)

    # Baseline answer
    start_time = time.perf_counter()
    baseline_answer, baseline_used_results = answer(
        q,
        retrieve_fn=baseline_search
    )
    baseline_time = time.perf_counter() - start_time

    print("\n--- BASELINE ---")
    print(f"Time: {baseline_time:.2f} seconds")
    print(baseline_answer)

    print("\nBaseline evidence:")
    show_results(q, baseline_used_results, label="Baseline")

    # Cross-encoder reranked answer
    start_time = time.perf_counter()
    reranked_answer, reranked_used_results = answer(
        q,
        retrieve_fn=crossencoder_rerank
    )
    reranked_time = time.perf_counter() - start_time

    print("\n--- CROSS-ENCODER RERANKED ---")
    print(f"Time: {reranked_time:.2f} seconds")
    print(reranked_answer)

    print("\nReranked evidence:")
    show_results(q, reranked_used_results, label="Cross-encoder reranked")

    print("\n--- EVALUATION ---")
    print("Fill in your lab table by checking:")
    print("- Is the top result relevant? (Y/N)")
    print("- Is the answer correct? (Y/Partial/N)")
    print("- Is every important claim supported by a citation? (Y/N)")
    print("- Did reranking improve the answer or only change the order?")


QUERY: What AI practices are prohibited under the EU AI Act?

--- BASELINE ---
Time: 2.95 seconds
The following AI practices are prohibited under the EU AI Act:

1. The placing on the market, putting into service, or use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective or effect of materially distorting the behavior of a person or a group of persons by appreciably impairing their ability to make an informed decision [source name, page 51.0].

2. Unfair commercial practices leading to economic or financial harms to consumers are prohibited under all circumstances, irrespective of whether they are implemented through AI systems or otherwise [source name, page 9.0].

Baseline evidence:

=== Baseline | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures 

---
## Step 8: Write `lab_proof.md` [YOUR TURN]

Create `lab_proof.md` in your repo (this is the **entry point for your reviewer**). Use this template:

```markdown
# Lab Proof: First match ≠ right match

## 1. What I built
One or two sentences, plus a link to this notebook.

## 2. Query
"..."

## 3. Retrieved evidence
| Rank | Source | Page / chunk_id | Score | Snippet |
|---|---|---|---|---|

## 4. Final answer
(paste the answer, including its citations)

## 5. Trace
Which claim in the answer comes from which chunk?

## 6. Baseline vs reranked
What changed? Was it worth the cost?

## 7. Failure or limitation
Where could this system give a plausible but unsupported answer?
```

**Before you submit:**
- [ ] Only files for this lab are in the repo
- [ ] No API keys, `.env` or tokens are committed
- [ ] `lab_proof.md` is the entry point
- [ ] A reviewer can trace **one** answer back to the exact retrieved chunk

---
### Extensions (optional)
- **Hybrid search:** combine BM25 keyword search with vector search, then rerank
- **Cost analysis:** cost per query for baseline vs LLM scoring vs cross-encoder
- **Failure test:** ask something neither source covers. Does your pipeline say "I don't know"?